# Notebook 03: Cluster Modeling

Explore KMeans location-hub clusters per pitcher + pitch type.

- Use the elbow method to evaluate k choices
- Fit and inspect cluster models
- Visualize location hubs on the strike zone

The clusters are NOT true intent — they are empirical location modes. Language throughout reflects this.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from src.services.pitcher_service import (
    get_pitcher_dataset, fit_pitch_type_clusters, get_available_pitch_types
)
from src.features.clustering import select_k_elbow, assign_clusters_to_game
from src.visualization.zone_plots import (
    prep_cluster_map, plot_cluster_map
)
from src import config

%matplotlib inline

In [ ]:
# ---- Parameters ----
PITCHER    = "Gerrit Cole"
START_DATE = "2024-04-01"
END_DATE   = "2024-09-30"
PITCH_TYPE = "FF"          # Four-seam fastball
K          = 4             # Number of clusters
# --------------------

df = get_pitcher_dataset(PITCHER, START_DATE, END_DATE)

## Available pitch types

In [ ]:
available = get_available_pitch_types(df)
print(f"Pitch types with >= {config.MIN_PITCHES_DEFAULT} pitches: {available}")

## Elbow method: choose k

In [ ]:
from src.data.fetch import resolve_pitcher_id
pitcher_id = resolve_pitcher_id(PITCHER)

elbow = select_k_elbow(df, pitcher_id, PITCH_TYPE, k_range=range(2, 8))
print("k -> inertia:")
for k, inertia in sorted(elbow.items()):
    print(f"  k={k}: {inertia:.1f}")

if elbow:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(list(elbow.keys()), list(elbow.values()), marker='o', color='steelblue')
    ax.set_xlabel("k (clusters)")
    ax.set_ylabel("Inertia")
    ax.set_title(f"Elbow method: {PITCHER} {PITCH_TYPE}")
    ax.grid(linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

## Fit baseline cluster model

In [ ]:
cluster_model = fit_pitch_type_clusters(
    df, pitcher_id, PITCH_TYPE, k=K
)

if cluster_model is None:
    print(f"Insufficient data for {PITCH_TYPE}")
else:
    print(f"Model: k={cluster_model.k}, n_pitches_fit={cluster_model.n_pitches_fit}")
    print(f"Inertia: {cluster_model.inertia:.2f}")
    print("Cluster centers (plate_x, plate_z):")
    for i, center in enumerate(cluster_model.centers):
        print(f"  Hub {i}: x={center[0]:.3f}, z={center[1]:.3f}")

## Location hub map (all pitches)

In [ ]:
if cluster_model is not None:
    # Assign clusters to all pitches in the dataset for visualization
    game_ids = df['game_pk'].unique() if 'game_pk' in df.columns else [None]
    first_game = int(game_ids[0])

    game_df = assign_clusters_to_game(df, first_game, PITCH_TYPE, cluster_model)
    prepped = prep_cluster_map(game_df, cluster_model.centers)
    fig = plot_cluster_map(
        prepped,
        title=f"{PITCHER} {PITCH_TYPE} — Location Hubs (k={K})"
    )
    plt.show()

## All pitch types: fit and summarize

In [ ]:
for pt in available:
    model = fit_pitch_type_clusters(df, pitcher_id, pt, k=K)
    if model:
        print(f"{pt}: k={model.k}, n={model.n_pitches_fit}, inertia={model.inertia:.1f}")
    else:
        print(f"{pt}: skipped (insufficient data)")